# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`: Aryan Daga**  
**`Roll Number`: U20230084**  
**`GitHub Branch`: aryan_U20230084**  

# Imports and Setup

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [5]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [7]:
# Data cleaning and basic preprocessing
news_df = news_df.copy()
train_users = train_users.copy()
test_users = test_users.copy()

# Handle missing values in news articles
text_cols = ["headline", "short_description", "authors"]
for col in text_cols:
    news_df[col] = news_df[col].fillna("")

# Parse dates; fill any invalid dates with the most frequent date
news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce")
if news_df["date"].isna().any():
    news_df["date"] = news_df["date"].fillna(news_df["date"].mode()[0])

# Ensure numeric columns in user data are clean
user_num_cols = ["age", "income", "clicks", "purchase_amount"]
for df in (train_users, test_users):
    df[user_num_cols] = df[user_num_cols].apply(pd.to_numeric, errors="coerce")
    df[user_num_cols] = df[user_num_cols].fillna(df[user_num_cols].median())
    df["label"] = df["label"].astype(str).str.strip().str.lower()

# Encode labels for later use
user_label_encoder = LabelEncoder()
train_users["label_enc"] = user_label_encoder.fit_transform(train_users["label"])
test_users["label_enc"] = user_label_encoder.transform(test_users["label"])

# Encode news categories (useful for bandit arm mapping)
news_category_encoder = LabelEncoder()
news_df["category_enc"] = news_category_encoder.fit_transform(news_df["category"].astype(str))

print("Missing values after cleaning:")
print("news_df:")
print(news_df.isna().sum())
print("train_users:")
print(train_users.isna().sum())


Missing values after cleaning:
news_df:
link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
category_enc         0
dtype: int64
train_users:
user_id            0
age                0
income             0
clicks             0
purchase_amount    0
label              0
label_enc          0
dtype: int64


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Feature matrix and labels
feature_cols = ["age", "income", "clicks", "purchase_amount"]
X_train = train_users[feature_cols]
y_train = train_users["label_enc"]
X_test = test_users[feature_cols]
y_test = test_users["label_enc"]

# Train a simple, reliable classifier
user_classifier = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

user_classifier.fit(X_train, y_train)

y_pred = user_classifier.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"User classification accuracy on test_users.csv: {acc:.4f}")
print(classification_report(y_test, y_pred, target_names=user_label_encoder.classes_))


User classification accuracy on test_users.csv: 0.3295
              precision    recall  f1-score   support

       user1       0.35      0.50      0.41       672
       user2       0.32      0.36      0.34       679
       user3       0.29      0.13      0.18       649

    accuracy                           0.33      2000
   macro avg       0.32      0.33      0.31      2000
weighted avg       0.32      0.33      0.31      2000



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
